In [15]:
import pandas as pd
from data_gatherer import polymarket_archive as pa

cfg = pa.load_config("data_gatherer/config.json")
day = pa.load_day(cfg["archive_dir"], "2026-01-10")

token_prices = pd.DataFrame(day["token_price_history"])   # minutely series
prices = pd.DataFrame(day["outcome_prices"])              # materialized snapshot prices
snaps = pd.DataFrame(day["market_snapshots"])

In [1]:
import pyarrow.parquet as pq

In [8]:
df = pq.read_table("/Users/gkbrkgrr/Desktop/polymarket-weather-trading/data/point_data/ecmwf-hres/raw/ecmwf-hres_2026011700_2026012100.parquet").to_pandas()

In [ ]:
grib_path = "/Users/gkbrkgrr/Desktop/polymarket-weather-trading/data/raster_data/gfs/2026011700/gfs_2026011700_f000.grib2"

In [18]:
import numpy as np
import xarray as xr

grib_path = "/Users/gkbrkgrr/Desktop/polymarket-weather-trading/data/raster_data/gfs/2026011700/gfs_2026011700_f000.grib2"

# Target location
LAT = 33.64
LON = -84.41  # West negative

# -------------------------
# Helper functions
# -------------------------
def open_cfgrib(path, filter_by_keys):
    return xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={"filter_by_keys": filter_by_keys},
    )

def normalize_lon(da, lon_name, lon):
    lon_vals = da[lon_name].values
    if np.nanmin(lon_vals) >= 0 and lon < 0:
        return (lon + 360.0) % 360.0
    return lon

def ensure_monotonic(da, lat_name, lon_name):
    out = da
    if not np.all(np.diff(out[lat_name].values) > 0):
        out = out.sortby(lat_name)
    if not np.all(np.diff(out[lon_name].values) > 0):
        out = out.sortby(lon_name)
    return out

def bilinear_point(da, lat_name, lon_name, lat, lon):
    da = ensure_monotonic(da, lat_name, lon_name)
    lon_use = normalize_lon(da, lon_name, lon)
    da_pt = da.interp({lat_name: lat, lon_name: lon_use}, method="linear")
    return da_pt, lon_use

# -------------------------
# Pressure-level temperature (t)
# -------------------------
ds_t = open_cfgrib(
    grib_path,
    {"typeOfLevel": "isobaricInhPa", "shortName": "t"},
)
t_pl = ds_t["t"]  # Kelvin

# -------------------------
# Pressure-level geopotential height (gh or z)
# -------------------------
try:
    ds_gh = open_cfgrib(
        grib_path,
        {"typeOfLevel": "isobaricInhPa", "shortName": "gh"},
    )
    gh = ds_gh["gh"]
except Exception:
    ds_gh = open_cfgrib(
        grib_path,
        {"typeOfLevel": "isobaricInhPa", "shortName": "z"},
    )
    gh = ds_gh["z"]

# -------------------------
# 2-meter temperature (t2m)
# -------------------------
ds_t2m = open_cfgrib(
    grib_path,
    {"typeOfLevel": "heightAboveGround", "shortName": "2t"},
)
t2m = ds_t2m["t2m"] if "t2m" in ds_t2m.data_vars else ds_t2m["2t"]  # Kelvin

# -------------------------
# Coordinate names
# -------------------------
lev = "isobaricInhPa" if "isobaricInhPa" in t_pl.coords else "level"
latn = "latitude" if "latitude" in t_pl.coords else "lat"
lonn = "longitude" if "longitude" in t_pl.coords else "lon"

latn2 = "latitude" if "latitude" in t2m.coords else "lat"
lonn2 = "longitude" if "longitude" in t2m.coords else "lon"

# -------------------------
# Optional: select single time/step if present
# -------------------------
for da in (t_pl, gh, t2m):
    if "time" in da.dims:
        da = da.isel(time=0)
    if "step" in da.dims:
        da = da.isel(step=0)

# -------------------------
# Bilinear interpolation
# -------------------------
t_pl_pt,  lon_used_pl  = bilinear_point(t_pl, latn,  lonn,  LAT, LON)
gh_pt,    _            = bilinear_point(gh,   latn,  lonn,  LAT, LON)
t2m_pt,   lon_used_2m  = bilinear_point(t2m,  latn2, lonn2, LAT, LON)

# -------------------------
# Geopotential → height (if needed)
# -------------------------
gh_units = str(gh_pt.attrs.get("units", "")).lower()
if gh_units in ("m**2 s**-2", "m2 s-2", "m^2 s^-2", "m^2/s^2"):
    gh_m = gh_pt / 9.80665
    gh_m.attrs["units"] = "m"
else:
    gh_m = gh_pt  # already gpm or m

# -------------------------
# Print results
# -------------------------
print(f"Interpolated point: lat={LAT}, lon={LON}")
print(f"Longitude used (PL): {lon_used_pl}, (t2m): {lon_used_2m}")
print("")
print(f"t2m = {float(t2m_pt.values):.2f} K")
print("")
print("Pressure-level profile (bilinear interpolation)")
print("Level (hPa) |   GH (m)   |   T (K)")
print("-----------------------------------")

levels = t_pl_pt[lev].values
for p in levels:
    gh_val = float(gh_m.sel({lev: p}).values)
    t_val  = float(t_pl_pt.sel({lev: p}).values)
    print(f"{int(p):>10} | {gh_val:9.1f} | {t_val:7.2f}")


Interpolated point: lat=33.64, lon=-84.41
Longitude used (PL): -84.41, (t2m): -84.41

t2m = 279.63 K

Pressure-level profile (bilinear interpolation)
Level (hPa) |   GH (m)   |   T (K)
-----------------------------------
      1000 |     144.4 |  280.93
       925 |     784.4 |  278.52
       850 |    1470.7 |  277.14
       700 |    3027.7 |  268.68


In [21]:
df[df["City"] == "Toronto"]

,City,InitTimeUTC,ValidTimeUTC,ValidTimeLocal,Timezone,LeadHour,StationElevation,Temperature2m,Temperature1000mb,Temperature925mb,Temperature850mb,Temperature700mb,TemperatureStation
231,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 00:00:00+00:00,2026-01-16T19:00:00-05:00,America/Toronto,0,173.1264,270,270,266,262,258,269
232,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 03:00:00+00:00,2026-01-16T22:00:00-05:00,America/Toronto,3,173.1264,271,271,267,264,257,270
233,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 06:00:00+00:00,2026-01-17T01:00:00-05:00,America/Toronto,6,173.1264,272,273,269,265,257,271
234,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 09:00:00+00:00,2026-01-17T04:00:00-05:00,America/Toronto,9,173.1264,272,274,270,265,256,271
235,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 12:00:00+00:00,2026-01-17T07:00:00-05:00,America/Toronto,12,173.1264,272,273,270,266,255,272
236,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 15:00:00+00:00,2026-01-17T10:00:00-05:00,America/Toronto,15,173.1264,273,273,269,266,255,272
237,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 18:00:00+00:00,2026-01-17T13:00:00-05:00,America/Toronto,18,173.1264,274,274,269,265,255,273
238,Toronto,2026-01-17 00:00:00+00:00,2026-01-17 21:00:00+00:00,2026-01-17T16:00:00-05:00,America/Toronto,21,173.1264,273,273,268,263,255,271
239,Toronto,2026-01-17 00:00:00+00:00,2026-01-18 00:00:00+00:00,2026-01-17T19:00:00-05:00,America/Toronto,24,173.1264,268,269,265,261,257,267
240,Toronto,2026-01-17 00:00:00+00:00,2026-01-18 03:00:00+00:00,2026-01-17T22:00:00-05:00,America/Toronto,27,173.1264,265,266,262,260,256,264
